In [1]:
#10.Data_Wrangling_RealEstate.ipynb
import pandas as pd
import numpy as np

# 1. Import dataset
df = pd.read_csv("RealEstate_Prices.csv")

print("First five records:")
print(df.head())

print("\nDataset shape:", df.shape)

# Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned columns:", df.columns.tolist())

# 2. Handle missing values
for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode()
            if not mode.empty:
                df[col] = df[col].fillna(mode[0])

print("\nMissing values:")
print(df.isnull().sum())

# 3. Merge additional dataset if available
# Example: Uncomment and modify if a neighborhood dataset exists.
# extra = pd.read_csv("Neighborhood_Data.csv")
# extra.columns = extra.columns.str.strip().str.lower()
# df = df.merge(extra, on="neighborhood", how="left")

# 4. Filter and subset data
if "sale_price" in df.columns:
    df = df[df["sale_price"] > 0]

if "property_type" in df.columns:
    print("\nProperty types:")
    print(df["property_type"].value_counts())

# 5. Encode categorical variables
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

df_encoded = pd.get_dummies(
    df, columns=categorical_cols, dtype=int
)

print("\nEncoded dataset:")
print(df_encoded.head())

# 6. Aggregate data
if "neighborhood" in df.columns and "sale_price" in df.columns:
    print("\nAverage sale price by neighborhood:")
    print(
        df.groupby("neighborhood")["sale_price"]
        .mean().sort_values(ascending=False)
    )

if "property_type" in df.columns and "sale_price" in df.columns:
    print("\nAverage price by property type:")
    print(df.groupby("property_type")["sale_price"].mean())

# 7. Detect and handle outliers using IQR
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count} potential outliers")

    # Cap extreme values instead of deleting records
    df[col] = df[col].clip(lower=lower, upper=upper)

# Export cleaned and encoded datasets
df.to_csv("Cleaned_RealEstate_Prices.csv", index=False)
df_encoded.to_csv("Encoded_RealEstate_Prices.csv", index=False)

print("\nData wrangling completed successfully.")
print("Cleaned dataset saved.")
print("Encoded dataset saved.")

First five records:
   Sale Price  Bedrooms  Bathrooms  Area Sq Ft Property Type Neighborhood  \
0      250000         3          2        1500         House        North   
1      315000         4          2        2100         House         West   
2      190000         2          1        1100     Apartment        South   
3      420000         5          3        2800         House         West   
4      275000         3          2        1600     Apartment        South   

    Sale Date  
0  2024-01-15  
1  2024-03-10  
2  2024-05-21  
3  2024-07-02  
4  2024-09-18  

Dataset shape: (6, 7)

Cleaned columns: ['sale_price', 'bedrooms', 'bathrooms', 'area_sq_ft', 'property_type', 'neighborhood', 'sale_date']

Missing values:
sale_price       0
bedrooms         0
bathrooms        0
area_sq_ft       0
property_type    0
neighborhood     0
sale_date        0
dtype: int64

Property types:
property_type
House        3
Apartment    2
Condo        1
Name: count, dtype: int64

Encoded datase